### Import Libraries

In [ ]:
import pandas as pd

from datasets import load_dataset

from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances

from sentence_transformers import SentenceTransformer

## Exercise

Get accustomed to the Revolutionary Crossroads data and `Dataset`s.

### RevRoad `Dataset` Locations:

- https://huggingface.co/RevolutionCrossroads
- https://huggingface.co/datasets/RevolutionCrossroads/loc_chronicling_america_1770-1810 (247 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/loc_chronicling_america_1770-1810_issues (707 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/nara_revolutionary_war_pension_files (9.4 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/nara_revolutionary_war_pension_files_PDFs (4.23 TB)
- https://huggingface.co/datasets/RevolutionCrossroads/si_us_revolutionary_era_collections (7.7 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/RevX_OCR_results/viewer/block_level (19.5 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/RevX_NER_Entity_Occurrences (939 MB)

In [ ]:
# Set up dataset stream
ocr_ds = load_dataset("RevolutionCrossroads/RevX_OCR_results",
                      name="page_level",
                      split="train",
                      streaming=True)
ocr_ds.features

In [ ]:
# For filtering data stream
def is_pension(example):
  return example["collection"] == "rev_war_pension"
def is_news(example):
  return example["collection"] == "chronicling_america"

In [ ]:
# Filter stream and get 100 rows of text
news_ds = ocr_ds.filter(is_news)
news_100_ds = news_ds.take(100)
news_100_text = list(news_100_ds["new_ocr_text"])

In [ ]:
# Go through dataset and split into sentences
doc_ids = []
docs = []
for idx,text in enumerate(news_100_text):
  for sent in text.split("\n"):
    doc_ids.append(idx)
    docs.append(sent)

In [ ]:
pd.DataFrame(zip(docs, doc_ids), columns=("sent", "id"))

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")

In [ ]:
# get document embeddings and group into 8 clusters
embeddings = model.encode(docs, batch_size=32)
kmeans = KMeans(n_clusters=8, random_state=1010)
cluster_labels = kmeans.fit_predict(embeddings)

In [ ]:
# print representative sentences for each cluster
for idx,center in enumerate(kmeans.cluster_centers_):
  print(idx)
  dists = euclidean_distances([center], embeddings)[0]
  medoid_idxs = dists.argsort()
  for idx in medoid_idxs[:5]:
    print("\t", docs[idx])

In [ ]:
# order docs by term
search_term = "unfair taxation"
search_term_emb = model.encode([search_term])
dists = euclidean_distances(search_term_emb, embeddings)[0]
medoid_idxs = dists.argsort()

for idx in medoid_idxs[:5]:
  print("\t", docs[idx])